# Dataset Comparison and Merge

This notebook compares the two raw dataset sources:

- `dataset/raw/PlantVillage`
- `dataset/raw/PlantVillage1`

It checks whether:

1. Both datasets expose the same class folders.
2. They contain duplicate images.
3. They can be merged into one combined dataset without copying exact
   duplicates twice.


## Current Observations

- Both datasets contain the same 15 class folders.
- `PlantVillage` contains 20,639 image files.
- `PlantVillage1` contains 20,639 image files.

The code below verifies those observations programmatically and then builds
a combined dataset in `dataset/raw/PlantVillage_combined`.


In [25]:
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass
from hashlib import sha256
from pathlib import Path
import shutil

import pandas as pd


In [26]:
BASE_DIR = Path.cwd().resolve().parent
RAW_DIR = BASE_DIR / 'dataset' / 'raw'
DATASET_A = RAW_DIR / 'PlantVillage'
DATASET_B = RAW_DIR / 'PlantVillage1'
COMBINED_DIR = RAW_DIR / 'PlantVillage_combined'

assert DATASET_A.exists(), DATASET_A
assert DATASET_B.exists(), DATASET_B

DATASET_A, DATASET_B, COMBINED_DIR


(PosixPath('/home/moeenuddin/Desktop/Deep_learning/RAGAge/RAG_Diagnostic_Agent/dataset/raw/PlantVillage'),
 PosixPath('/home/moeenuddin/Desktop/Deep_learning/RAGAge/RAG_Diagnostic_Agent/dataset/raw/PlantVillage1'),
 PosixPath('/home/moeenuddin/Desktop/Deep_learning/RAGAge/RAG_Diagnostic_Agent/dataset/raw/PlantVillage_combined'))

In [27]:
def list_classes(dataset_dir: Path) -> list[str]:
    """Return sorted class-folder names for a dataset root."""
    return sorted(path.name for path in dataset_dir.iterdir() if path.is_dir())


classes_a = list_classes(DATASET_A)
classes_b = list_classes(DATASET_B)

comparison = pd.DataFrame(
    {
        'PlantVillage': pd.Series(classes_a),
        'PlantVillage1': pd.Series(classes_b),
    }
)
comparison


,PlantVillage,PlantVillage1
0,Pepper__bell___Bacterial_spot,Pepper__bell___Bacterial_spot
1,Pepper__bell___healthy,Pepper__bell___healthy
2,Potato___Early_blight,Potato___Early_blight
3,Potato___Late_blight,Potato___Late_blight
4,Potato___healthy,Potato___healthy
5,Tomato_Bacterial_spot,Tomato_Bacterial_spot
6,Tomato_Early_blight,Tomato_Early_blight
7,Tomato_Late_blight,Tomato_Late_blight
8,Tomato_Leaf_Mold,Tomato_Leaf_Mold
9,Tomato_Septoria_leaf_spot,Tomato_Septoria_leaf_spot


In [28]:
same_classes = classes_a == classes_b
only_in_a = sorted(set(classes_a) - set(classes_b))
only_in_b = sorted(set(classes_b) - set(classes_a))

print(f'Same class folders: {same_classes}')
print(f'Only in PlantVillage: {only_in_a}')
print(f'Only in PlantVillage1: {only_in_b}')
print(f'Total classes: {len(classes_a)}')


Same class folders: True
Only in PlantVillage: []
Only in PlantVillage1: []
Total classes: 15


In [29]:
def iter_files(dataset_dir: Path) -> list[Path]:
    """Return all files inside a dataset root in sorted order."""
    return sorted(path for path in dataset_dir.rglob('*') if path.is_file())


files_a = iter_files(DATASET_A)
files_b = iter_files(DATASET_B)

print(f'PlantVillage files: {len(files_a):,}')
print(f'PlantVillage1 files: {len(files_b):,}')


PlantVillage files: 20,639
PlantVillage1 files: 20,639


## Duplicate Detection

The next cells hash every file with SHA-256 and compare images across both
dataset roots. This detects exact duplicates even if filenames differ.


In [30]:
@dataclass(frozen=True)
class FileRecord:
    dataset: str
    class_name: str
    relative_path: str
    file_name: str
    sha256: str
    size_bytes: int


def file_hash(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Compute a stable SHA-256 hash for a file."""
    digest = sha256()
    with path.open('rb') as file_obj:
        while True:
            chunk = file_obj.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def build_records(dataset_dir: Path) -> list[FileRecord]:
    """Create file records with hashes for a dataset root."""
    records: list[FileRecord] = []
    for file_path in iter_files(dataset_dir):
        relative = file_path.relative_to(dataset_dir)
        records.append(
            FileRecord(
                dataset=dataset_dir.name,
                class_name=relative.parts[0],
                relative_path=relative.as_posix(),
                file_name=file_path.name,
                sha256=file_hash(file_path),
                size_bytes=file_path.stat().st_size,
            )
        )
    return records


In [31]:
records_a = build_records(DATASET_A)
records_b = build_records(DATASET_B)

df_a = pd.DataFrame(record.__dict__ for record in records_a)
df_b = pd.DataFrame(record.__dict__ for record in records_b)

duplicate_pairs = df_a.merge(
    df_b,
    on='sha256',
    suffixes=('_a', '_b'),
)

print(f'Exact duplicate matches across datasets: {len(duplicate_pairs):,}')
duplicate_pairs.head()


Exact duplicate matches across datasets: 20,667


,dataset_a,class_name_a,relative_path_a,file_name_a,sha256,size_bytes_a,dataset_b,class_name_b,relative_path_b,file_name_b,size_bytes_b
0,PlantVillage,Pepper__bell___Bacterial_spot,Pepper__bell___Bacterial_spot/0022d6b7-d47c-4e...,0022d6b7-d47c-4ee2-ae9a-392a53f48647___JR_B.Sp...,a3ff41a119c785e4e37683880c47456ae2a4c27f375380...,18052,PlantVillage1,Pepper__bell___Bacterial_spot,Pepper__bell___Bacterial_spot/0022d6b7-d47c-4e...,0022d6b7-d47c-4ee2-ae9a-392a53f48647___JR_B.Sp...,18052
1,PlantVillage,Pepper__bell___Bacterial_spot,Pepper__bell___Bacterial_spot/006adb74-934f-44...,006adb74-934f-448f-a14f-62181742127b___JR_B.Sp...,84a7cceea14c879fcfea25b52b90bfc0e8a13f1fbceb0f...,21211,PlantVillage1,Pepper__bell___Bacterial_spot,Pepper__bell___Bacterial_spot/006adb74-934f-44...,006adb74-934f-448f-a14f-62181742127b___JR_B.Sp...,21211
2,PlantVillage,Pepper__bell___Bacterial_spot,Pepper__bell___Bacterial_spot/00f2e69a-1e56-41...,00f2e69a-1e56-412d-8a79-fdce794a17e4___JR_B.Sp...,d1b01e4214316f9fb31bb84258cfd2487f3d7c0558e657...,18554,PlantVillage1,Pepper__bell___Bacterial_spot,Pepper__bell___Bacterial_spot/00f2e69a-1e56-41...,00f2e69a-1e56-412d-8a79-fdce794a17e4___JR_B.Sp...,18554
3,PlantVillage,Pepper__bell___Bacterial_spot,Pepper__bell___Bacterial_spot/01613cd0-d3cd-4e...,01613cd0-d3cd-4e96-945c-a312002037bf___JR_B.Sp...,87f35bce91e6b854a292f9d2a0e290772b576c6270e2ce...,26496,PlantVillage1,Pepper__bell___Bacterial_spot,Pepper__bell___Bacterial_spot/01613cd0-d3cd-4e...,01613cd0-d3cd-4e96-945c-a312002037bf___JR_B.Sp...,26496
4,PlantVillage,Pepper__bell___Bacterial_spot,Pepper__bell___Bacterial_spot/0169b9ac-07b9-4b...,0169b9ac-07b9-4be1-8b85-da94481f05a4___NREC_B....,af40e0a37aa733df6f9be4976e692efc003edcc155ddcd...,19517,PlantVillage1,Pepper__bell___Bacterial_spot,Pepper__bell___Bacterial_spot/0169b9ac-07b9-4b...,0169b9ac-07b9-4be1-8b85-da94481f05a4___NREC_B....,19517


In [32]:
duplicates_by_class = (
    duplicate_pairs.groupby('class_name_a').size().sort_values(ascending=False)
)
duplicates_by_class.rename('duplicate_count').to_frame()


,duplicate_count
class_name_a,
Tomato__Tomato_YellowLeaf__Curl_Virus,3209
Tomato_Bacterial_spot,2127
Tomato_Late_blight,1925
Tomato_Septoria_leaf_spot,1771
Tomato_Spider_mites_Two_spotted_spider_mite,1676
Tomato_healthy,1603
Pepper__bell___healthy,1478
Tomato__Target_Spot,1404
Potato___Early_blight,1000


## Merge Into a Combined Dataset

This merge keeps all unique files and skips exact duplicates. If a duplicate
hash already exists in the combined output for a class, the file is not copied
again. If a filename collides but the content is different, a suffix is added.


In [33]:
def copy_unique_files(
    source_dir: Path,
    target_dir: Path,
    seen_hashes: dict[str, set[str]],
) -> Counter:
    """Copy files into a class-aligned target directory without duplicates."""
    stats: Counter = Counter()
    for file_path in iter_files(source_dir):
        relative = file_path.relative_to(source_dir)
        class_name = relative.parts[0]
        file_digest = file_hash(file_path)
        class_target_dir = target_dir / class_name
        class_target_dir.mkdir(parents=True, exist_ok=True)

        if file_digest in seen_hashes.setdefault(class_name, set()):
            stats['skipped_duplicates'] += 1
            continue

        destination = class_target_dir / file_path.name
        if destination.exists():
            stem = destination.stem
            suffix = destination.suffix
            index = 1
            while destination.exists():
                destination = class_target_dir / f'{stem}_{index}{suffix}'
                index += 1

        shutil.copy2(file_path, destination)
        seen_hashes[class_name].add(file_digest)
        stats['copied_files'] += 1

    return stats


In [34]:
if COMBINED_DIR.exists():
    raise FileExistsError(
        f'{COMBINED_DIR} already exists. Remove it first if you want a fresh merge.'
    )

seen_hashes: dict[str, set[str]] = {}
stats_a = copy_unique_files(DATASET_A, COMBINED_DIR, seen_hashes)
stats_b = copy_unique_files(DATASET_B, COMBINED_DIR, seen_hashes)

merge_summary = pd.DataFrame(
    [
        {'dataset': DATASET_A.name, **stats_a},
        {'dataset': DATASET_B.name, **stats_b},
    ]
).fillna(0)
merge_summary


,dataset,copied_files,skipped_duplicates
0,PlantVillage,20625.0,14
1,PlantVillage1,0.0,20639


In [35]:
combined_counts = {
    class_dir.name: len([path for path in class_dir.iterdir() if path.is_file()])
    for class_dir in sorted(COMBINED_DIR.iterdir())
    if class_dir.is_dir()
}

pd.Series(combined_counts, name='combined_file_count').sort_index().to_frame()


,combined_file_count
Pepper__bell___Bacterial_spot,997
Pepper__bell___healthy,1478
Potato___Early_blight,1000
Potato___Late_blight,1000
Potato___healthy,152
Tomato_Bacterial_spot,2127
Tomato_Early_blight,1000
Tomato_Late_blight,1901
Tomato_Leaf_Mold,952
Tomato_Septoria_leaf_spot,1771


In [36]:
print(f'Combined dataset created at: {COMBINED_DIR}')
print(f'Total combined files: {sum(combined_counts.values()):,}')


Combined dataset created at: /home/moeenuddin/Desktop/Deep_learning/RAGAge/RAG_Diagnostic_Agent/dataset/raw/PlantVillage_combined
Total combined files: 20,625
